# Calculate 1$^\circ$-based parameters and aggregate to shapefiles
For resolution testing.

For main analysis / figures, all using 1-deg QDM runs:
- QPLAD `tassumpoly` to impact regions 
- QPLAD `tassumpoly` to ADM1s
- QPLAD `tasmaxbinF` to US counties
- Will need to calculate `tassumpoly`/`tasmaxbinF` for 1-deg QDM runs first

In [1]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
from numba import jit as njit
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar

import distributed

from funcs_support import get_filepaths,get_params,utility_save
dir_list = get_params()

In [2]:
from distributed import Client

# Start dask client 
# (playing around with workers, threads for efficiency)
client = Client(n_workers=6,
    threads_per_worker=3,
    processes=True,
    memory_limit="8GB")
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/8787/status,Workers: 6
Total threads: 18,Total memory: 44.70 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34757,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:42117,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/46425/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:34587,


In [3]:
# Get paths of BCD-ME files
bcd_fns = pd.DataFrame({'path':np.sort(glob.glob(dir_list['bcd_me']+'*.zarr'))})
bcd_fns['model'] = [re.split(r'\.',re.split(r'\_',fn)[-1])[0] for fn in bcd_fns.path]
bcd_fns['typ'] = [re.split(r'_',re.split(r'\/',fn)[-1])[2] for fn in bcd_fns.path.values]
modlist = bcd_fns['model'].unique()
bcd_fns = bcd_fns.set_index(['typ','model'])

In [4]:
params_proc = {'impact-regions':{'varnames':['tassumpoly'],'shp_fn':'impact-region'},
               'adm1':{'varnames':['tassumpoly'],'shp_fn':'country-fixed'},
               'adm2-us':{'varnames':['tasmaxbinF'],'shp_fn':'UScounties'}}


idxs = {'impact-regions':{'hierid':'locid'},
        'adm1':{'CNTRY_NAME':'locid'},
        'adm2-us':{'FIPS':'locid'}}

In [5]:
# Load population density
pop = xr.open_dataset(dir_list['aux']+'pop_data/glp00ag30.asc')
pop = xa.fix_ds(pop)
pop = pop.isel(band=0).band_data

# To get around an xagg error when weights are exactly 0
pop = pop.where(pop>0,0.0000000001) 

In [6]:
@njit
def numba_expsums(data,degs,C_from_K = False):
    ''' Numbaized sum of exponents function
    '''

    sums = np.full(len(degs),np.nan)

    if C_from_K:
        data = data-273.15

    if not np.all(np.isnan(data)):
        for i,deg in enumerate(degs):
            sums[i] = sum(data**deg)

    return sums
    

def dmgf_params_carleton(da,degs,dim='dayofyear'):
    ''' Calculate necessary parameters for Carleton et al. Damage Function

    Get T, T^2, T^3, T^4 for each location

    Compare to the equation in B.2.4 (Supp Material), which is 
       T_it = [ Sum_day ( Sum_pix ( w_pix (T_pix,day)^k ) ) for k in 1, 2, 3, 4]

    Since summations are commutative, I change this to 
       T_it = [ Sum_pix ( w_pix * Sum _day (T_pix,day)^k ) for k in 1, 2, 3, 4]

    Parameters
    ---------------
    da : :py:meth:`xr.DataArray`

    degs : list / array
        Exponential degrees (the ks in T**k), e.g. [1,2,3,4]
        
    dim : :py:meth:`str`, by default 'dayofyear'
        The dimension over which to conduct the histogram 

    Returns
    ---------------
    params : :py:meth:`xr.DataArray` 
        With the daysofyear dimension replaced with n_params (in this case 4)
    
    '''

    params = xr.apply_ufunc(
        numba_expsums,
        da,
        input_core_dims=[[dim]],
        output_core_dims=[['param']],
        kwargs={'degs': degs},
        vectorize=True,
        dask="parallelized",
        output_dtypes=[np.float32],
        dask_gufunc_kwargs = {'output_sizes':{'param':len(degs)}}
    )
    params['param'] = ['T'+str(k) for k in degs]
    return params


@njit
def numba_histogram(data, bins, edges = 'inf'):
    ''' Numbaized histogram function

    '''
    if edges == 'inf':
        bins = np.array([-np.inf,*bins,np.inf])
        
    out_len = len(bins) - 1

    # int16 more than sufficient for max value of 365
    hist = np.zeros(out_len, dtype=np.int16)
    if not np.all(np.isnan(data)):
        for value in data:
            for i in range(out_len):
                if bins[i] <= value < bins[i + 1]:
                    hist[i] += 1
                    break
    return hist

def dmgf_params_bins(da, bins, dim='dayofyear'):
    ''' Generate histogram bins

    Parameters
    ---------------
    da : :py:meth:`xr.DataArray`

    bins : iterable
        The bins of the histogram

    dim : :py:meth:`str`, by default 'dayofyear'
        The dimension over which to conduct the histogram

    Returns
    ---------------
    bins_out : :py:meth:`xr.DataArray` 
        With the `dim` dimension replaced with n_bins

    '''
    bins_out =  xr.apply_ufunc(
            numba_histogram,
            da,
            input_core_dims=[[dim]],
            output_core_dims=[['bin']],
            kwargs={'bins': bins},
            vectorize=True,
            dask="parallelized",
            output_dtypes=[np.int16],
            dask_gufunc_kwargs = {'output_sizes':{'bin':len(bins)+1}}
        )
    bins_out['bin'] = np.array([bins[0] - np.mean(np.diff(bins))/2,
                               *(bins[0:-1] + np.diff(bins)/2),
                               bins[-1] + np.mean(np.diff(bins))/2])
    
    return bins_out

## Calculate regression parameters for 1-degree QDM files 

In [6]:
bcd_fns.loc['qdm']

,path
model,
ACCESS-ESM1-5,/glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd...
CESM2-WACCM,/glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd...
CNRM-CM6-1,/glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd...
CNRM-ESM2-1,/glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd...
CanESM5,/glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd...
EC-Earth3-Veg,/glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd...
EC-Earth3,/glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd...
FGOALS-g3,/glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd...
IPSL-CM6A-LR,/glade/campaign/cgd/cas/schwarzwald/bcd_me/bcd...


In [ ]:
# Load sample QDM 1-degree file
ds_grid = xr.open_zarr(bcd_fns.loc['qdm'].iloc[0]['path'],decode_timedelta=False)[['lat','lon']].load()
# Weird xagg read-only error in xa.auxfuncs.get_bnds, manually setting bounds should sidestep it
# (lat bounds already included) (this is a problem with the `impact_uncertainty` environment, but not the `bcd_me` one, hmmm...)
ds_grid['lon_bnds'] = xr.concat([ds_grid.lon-0.5,
           ds_grid.lon+0.5],
          dim='bnds').transpose('lon','bnds')

single_poly = gdf.union_all()
ds_grid = ds_grid.sel(lat=slice(single_poly.bounds[1]-1,single_poly.bounds[3]+1),
       lon=slice(single_poly.bounds[0]-1,single_poly.bounds[2]+1))

for proc in params_proc:
    #------------------------
    # Setup -----------------
    #------------------------
    # Generate filenames
    output_fns = {mod:{var:{'nowght':f'{dir_list['proc']}aggs/{var}_agg-1deg_{mod}_ALLRUNS_ALLGWLs_{proc}.zarr',
                            'popwght':f'{dir_list['proc']}aggs/{var}_agg-1deg_{mod}_ALLRUNS_ALLGWLs_{proc}_popwght.zarr'}
                   for var in params_proc[proc]['varnames']}
             for mod in modlist}
    
    # Check for completed existence of all files for a given 
    # shapefile (utility_print adds an empty .done flag after saving)
    if np.all([np.all([np.all([os.path.exists(output_fns[k][v][t]+'/.done') 
                               for t in output_fns[k][v]]) 
                       for v in output_fns[k]]) 
               for k in output_fns.keys()]):
        print(f'All files generated for {proc}, skipped.')
        #continue
    
    #------------------------
    # Generate weightmaps ---
    #------------------------
    # Load shapefile
    gdf = gpd.read_file(f'{dir_list['aux']}geo_data/{params_proc[proc]['shp_fn']}.shp')
    gdf['geometry'] = gdf.make_valid()
    
    # Keep relevant index columns
    gdf = gdf.loc[:,[*[k for k in idxs[proc]],'geometry']]
    
    # Rename index columns to standard names across analyses
    gdf = gdf.rename(idxs[proc],axis=1)
    
    # Create weightmap
    with warnings.catch_warnings():
        # Catches a weights bbox warning that doesn't affect this process
        warnings.filterwarnings('ignore')
        wms = {'nowght':xa.pixel_overlaps(ds_grid,gdf),
               'popwght':xa.pixel_overlaps(ds_grid,gdf,weights=pop)}


    #------------------------
    # Aggregate by model ----
    #------------------------
    for mod in tqdm(modlist):
        for var in params_proc[proc]['varnames']:
            # Check for completed existence of files for a given
            # model / variable combination (utility_print adds an
            # empty .done flag after saving)
            path_exists = {wght:os.path.exists(output_fns[mod][var][wght]+'/.done') for wght in output_fns[mod][var]}
            if np.all([v for k,v in path_exists.items()]):
                print(f'All files for {mod} {var} aggregated to {proc} already generated, skipped.')
                #continue
            proc_weights = [wght for wght,flag in path_exists.items() if not flag]
            
            # Load QDM 1-deg bias-corrected data
            if re.search(r'^tasmax',var):
                basevar = 'tasmax'
            else:
                basevar = 'tas'
            
            ds = xr.open_zarr(bcd_fns.loc[('qdm',mod),'path'],decode_timedelta=False)
            has_data = ds.has_data.load()
            if basevar in ds:
                ds = ds[[basevar]]
            else:
                print(f'{basevar} not in {bcd_fns.loc[('qdm',mod),'path']}, skipped.')
                #continue
            
            # Limit to bounding box of shapefiles (useful mainly for the US counties one)
            ds = ds.sel(lat=slice(single_poly.bounds[1]-1,single_poly.bounds[3]+1),
                   lon=slice(single_poly.bounds[0]-1,single_poly.bounds[2]+1))
            
            ds = ds.sel(proj_base = ['ERA5','MERRA2','JRA-3Q','GMFD'])
            
            # Rechunk to allow geographic processing
            #ds = ds.chunk({'lat':-1,'lon':-1,'proj_base':-1})
            ds = ds.chunk({'dayofyear':-1,'year':-1})

            
            #------------------------
            # Calculate params ------
            #------------------------
            if not os.path.exists(fn_tmp_params+'/.done'):
                # Create 
                if 'sumpoly' in var:
                    # Calculate sums of polynomials, used e.g. by the Carleton et al.
                    # damage function, make sure it's in C if a temperature variable
                    ds_tmp = (dmgf_params_carleton((ds[basevar]-
                                                        ('tas' in basevar)*273.15), 
                                                       np.arange(1,5)).
                                  to_dataset(name=basevar+'sumpoly'))
                    ds_tmp = ds_tmp.mean('year')
                elif 'binF' in var:
                    binsF = np.arange(0, 130, 5).astype(float) 
                    # Calculate days in bins (converting to F from K)
                    ds_tmp = (dmgf_params_bins(ds[basevar]*(9/5) - 459.67, bins=binsF).
                                to_dataset(name=basevar+'binF'))
                    ds_tmp = ds_tmp.mean('year')
                
                # Save temporary file for damage function parameters
                fn_tmp_params = f'/glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/{var}_1deg_{mod}_TMP.zarr'
                # Ensure has_data is kept; it occasionally gets lost for some reason
                ds_tmp = ds_tmp.assign_coords({'has_data':has_data})
                with warnings.catch_warnings():
                    # Zarr metadata warnings, etc.
                    warnings.filterwarnings('ignore')
                    ds_tmp.to_zarr(fn_tmp_params,consolidated=False,mode='w')
                    open(fn_tmp_params+'/.done', 'w').close()
                print(f'Temp file {fn_tmp_params} saved')
            else:
                print(f'Temp file {fn_tmp_params} loaded!')

            
            #------------------------
            # Aggregate -------------
            #------------------------
            ds_tmp = xr.open_zarr(fn_tmp_params)
            # Chunk for geographic processing
            ds_tmp = ds_tmp.chunk({'idv':1,'lat':-1,'lon':-1})
            # Aggregate
            with warnings.catch_warnings():
                # Stops inconsistent nan warning from xagg - which is just catching that
                # some gwls are all-nans for some idvs
                warnings.filterwarnings('ignore')
                
                with xa.set_options(impl='numba',silent=True):
                    aggs = {wght:xa.aggregate(ds_tmp,wms[wght])
                            for wght in proc_weights}
            
            # Move aggregated output to xarray dataset format
            aggs_out = {wght:agg.to_dataset()
                        for wght,agg in aggs.items()}
                    
            # Add experiment / run id back (secondary coordinates don't make it
            # through the xagg workflow) 
            for wght in aggs_out:
                aggs_out[wght] = aggs_out[wght].reset_index('idv').assign_coords({var:ds_tmp[var] for var in ['experiment','run','model','has_data']})

            #------------------------
            # Save ----
            #------------------------
            for wght in aggs_out:
                # Metadata
                if wght == 'nowght':
                    weightdesc = ''
                elif wght == 'popwght':
                    weightdesc = ' using pop. weights'
                aggs_out[wght].attrs = {'SOURCE':'aggregate_to_shps.ipynb',
                                        'DESCRIPTION':f'BCD-ME 1-degree {mod} {var} data, aggregated{weightdesc} to {re.sub(r'\-', ' ',proc)} polygons'}
            
                # Chunking for zarr saving
                chunk_dims = ['idv','proj_base']
                full_dims = [dim for dim in aggs_out[wght][var].sizes if dim not in chunk_dims]
                aggs_out[wght] = aggs_out[wght].chunk({**{dim:1 for dim in chunk_dims},
                          **{dim:-1 for dim in full_dims}}
                        )
            
                # Output as zarr
                with warnings.catch_warnings():
                    # Zarr metadata warnings, etc.
                    warnings.filterwarnings('ignore')
                    utility_save(aggs_out[wght].drop_encoding(),
                                 output_fns[mod][var][wght],
                                 zarr_mode = 'w',
                                 save_kwargs = {'zarr_format':3})
                
            # Remove temporary file
            os.system('rm -rf '+fn_tmp_params)
            print('Temporary file '+fn_tmp_params+' removed!')

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: /glade/u/home/schwarzwald/projects/impact_uncertainty/aux_data/geo_data/impact-region.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


creating polygons for each pixel...
lat/lon bounds not found in dataset; they will be created.
calculating overlaps between pixels and output polygons...
success!
creating polygons for each pixel...
regridding weights to data grid...
calculating overlaps between pixels and output polygons...
success!


  0%|          | 0/12 [00:00<?, ?it/s]

Temp file /glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/tassumpoly_1deg_ACCESS-ESM1-5_TMP.zarr saved


/glade/derecho/scratch/schwarzwald/tmp/ipykernel_120540/1000576077.py:129: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds_tmp = xr.open_zarr(fn_tmp_params)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


...

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_CESM2-WACCM_ALLRUNS_ALLGWLs_adm1.zarr saved!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_CESM2-WACCM_ALLRUNS_ALLGWLs_adm1_popwght.zarr saved!
Temporary file /glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/tassumpoly_1deg_CESM2-WACCM_TMP.zarr removed!
Temp file /glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/tassumpoly_1deg_CNRM-CM6-1_TMP.zarr saved


/glade/derecho/scratch/schwarzwald/tmp/ipykernel_120540/1000576077.py:129: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds_tmp = xr.open_zarr(fn_tmp_params)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_CNRM-CM6-1_ALLRUNS_ALLGWLs_adm1.zarr saved!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_CNRM-CM6-1_ALLRUNS_ALLGWLs_adm1_popwght.zarr saved!
Temporary file /glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/tassumpoly_1deg_CNRM-CM6-1_TMP.zarr removed!
Temp file /glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/tassumpoly_1deg_CNRM-ESM2-1_TMP.zarr saved


/glade/derecho/scratch/schwarzwald/tmp/ipykernel_120540/1000576077.py:129: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  ds_tmp = xr.open_zarr(fn_tmp_params)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/group.py:3535: ZarrUserWarning: Object at .done is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_CNRM-ESM2-1_ALLRUNS_ALLGWLs_adm1.zarr saved!
/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_CNRM-ESM2-1_ALLRUNS_ALLGWLs_adm1_popwght.zarr saved!
Temporary file /glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/tassumpoly_1deg_CNRM-ESM2-1_TMP.zarr removed!


In [11]:
mod = 'ACCESS-ESM1-5'
var = 'tassumpoly'

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=9, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in

In [93]:
ds_tmp.chunk({'idv':1,'lat':-1,'lon':-1}).tassumpoly

<xarray.DataArray 'tassumpoly' (idv: 25, proj_base: 4, lat: 141, lon: 360,
                                gwl: 8, param: 4)> Size: 650MB
dask.array<xarray-tassumpoly, shape=(25, 4, 141, 360, 8, 4), dtype=float32, chunksize=(1, 4, 141, 360, 8, 4), chunktype=numpy.ndarray>
Coordinates:
    model       (idv) <U13 1kB dask.array<chunksize=(1,), meta=np.ndarray>
    experiment  (idv) <U6 600B dask.array<chunksize=(1,), meta=np.ndarray>
    run         (idv) <U9 900B dask.array<chunksize=(1,), meta=np.ndarray>
  * proj_base   (proj_base) <U6 96B 'ERA5' 'MERRA2' 'JRA-3Q' 'GMFD'
  * lat         (lat) float64 1kB -55.5 -54.5 -53.5 -52.5 ... 82.5 83.5 84.5
  * lon         (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * gwl         (gwl) float64 64B 0.61 1.0 1.5 2.0 2.5 3.0 3.5 4.0
  * param       (param) <U2 32B 'T1' 'T2' 'T3' 'T4'
Dimensions without coordinates: idv
Attributes:
    long_name:      Daily Near-Surface Air Temperature
    standard_name:  air_temperature
    units:          K

/glade/u/home/schwarzwald/projects/impact_uncertainty/code/funcs_support.py:447: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_zarr(output_fn,**save_kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type 

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_adm1.zarr saved!


/glade/u/home/schwarzwald/projects/impact_uncertainty/code/funcs_support.py:447: SerializationWarning: variable None has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_zarr(output_fn,**save_kwargs)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type 

/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_adm1_popwght.zarr saved!


In [87]:
output_fns

{'ACCESS-ESM1-5': {'tassumpoly': {'nowght': '/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_adm1.zarr',
   'popwght': '/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_ACCESS-ESM1-5_ALLRUNS_ALLGWLs_adm1_popwght.zarr'}},
 'CESM2-WACCM': {'tassumpoly': {'nowght': '/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_CESM2-WACCM_ALLRUNS_ALLGWLs_adm1.zarr',
   'popwght': '/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_CESM2-WACCM_ALLRUNS_ALLGWLs_adm1_popwght.zarr'}},
 'CNRM-CM6-1': {'tassumpoly': {'nowght': '/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_CNRM-CM6-1_ALLRUNS_ALLGWLs_adm1.zarr',
   'popwght': '/glade/derecho/scratch/schwarzwald/impact_uncertainty/aggs/tassumpoly_agg-1deg_CNRM-CM6-1_ALLRUNS_ALLGWLs_adm1_popwght.zarr'}},
 'CNRM-ESM2-1': {'tassumpoly': {'nowght': '/glade/derecho/scratch/schwarzwald/i

In [79]:
ds_tmp.assign_coords({'has_data':has_data})

<xarray.Dataset> Size: 650MB
Dimensions:     (idv: 25, proj_base: 4, lat: 141, lon: 360, gwl: 8, param: 4,
                 variable: 2)
Coordinates:
    experiment  (idv) <U6 600B 'ssp245' 'ssp245' 'ssp245' ... 'ssp585' 'ssp585'
    model       (idv) <U13 1kB 'ACCESS-ESM1-5' ... 'ACCESS-ESM1-5'
    run         (idv) <U9 900B 'r10i1p1f1' 'r11i1p1f1' ... 'r5i1p1f1' 'r9i1p1f1'
  * proj_base   (proj_base) <U6 96B 'ERA5' 'MERRA2' 'JRA-3Q' 'GMFD'
  * lat         (lat) float64 1kB -55.5 -54.5 -53.5 -52.5 ... 82.5 83.5 84.5
  * lon         (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * gwl         (gwl) float64 64B 0.61 1.0 1.5 2.0 2.5 3.0 3.5 4.0
    has_data    (variable, idv, gwl) bool 400B True True True ... True True True
  * param       (param) <U2 32B 'T1' 'T2' 'T3' 'T4'
  * variable    (variable) object 16B 'tas' 'tasmax'
Dimensions without coordinates: idv
Data variables:
    tassumpoly  (idv, proj_base, lat, lon, gwl, param) float32 650MB ...

In [81]:
ds_tmp.assign_coords({'has_data':has_data}).to_zarr(re.sub(r'\.zarr','2.zarr',fn_tmp_params),mode='w')

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=9, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in

In [71]:
print(ds_tmp)

<xarray.Dataset> Size: 650MB
Dimensions:     (idv: 25, proj_base: 4, lat: 141, lon: 360, gwl: 8, param: 4)
Coordinates:
    run         (idv) <U9 900B dask.array<chunksize=(25,), meta=np.ndarray>
    experiment  (idv) <U6 600B dask.array<chunksize=(25,), meta=np.ndarray>
    model       (idv) <U13 1kB dask.array<chunksize=(25,), meta=np.ndarray>
  * proj_base   (proj_base) <U6 96B 'ERA5' 'MERRA2' 'JRA-3Q' 'GMFD'
  * lat         (lat) float64 1kB -55.5 -54.5 -53.5 -52.5 ... 82.5 83.5 84.5
  * lon         (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * gwl         (gwl) float64 64B 0.61 1.0 1.5 2.0 2.5 3.0 3.5 4.0
  * param       (param) <U2 32B 'T1' 'T2' 'T3' 'T4'
Dimensions without coordinates: idv
Data variables:
    tassumpoly  (idv, proj_base, lat, lon, gwl, param) float32 650MB dask.array<chunksize=(1, 1, 40, 40, 1, 4), meta=np.ndarray>


In [82]:
xr.open_zarr(re.sub(r'\.zarr','2.zarr',fn_tmp_params))

<xarray.Dataset> Size: 650MB
Dimensions:     (idv: 25, proj_base: 4, lat: 141, lon: 360, gwl: 8, param: 4,
                 variable: 2)
Coordinates:
    experiment  (idv) <U6 600B dask.array<chunksize=(25,), meta=np.ndarray>
    model       (idv) <U13 1kB dask.array<chunksize=(25,), meta=np.ndarray>
    run         (idv) <U9 900B dask.array<chunksize=(25,), meta=np.ndarray>
  * proj_base   (proj_base) <U6 96B 'ERA5' 'MERRA2' 'JRA-3Q' 'GMFD'
  * lat         (lat) float64 1kB -55.5 -54.5 -53.5 -52.5 ... 82.5 83.5 84.5
  * lon         (lon) float64 3kB -179.5 -178.5 -177.5 ... 177.5 178.5 179.5
  * gwl         (gwl) float64 64B 0.61 1.0 1.5 2.0 2.5 3.0 3.5 4.0
    has_data    (variable, idv, gwl) bool 400B dask.array<chunksize=(2, 25, 8), meta=np.ndarray>
  * param       (param) <U2 32B 'T1' 'T2' 'T3' 'T4'
  * variable    (variable) object 16B 'tas' 'tasmax'
Dimensions without coordinates: idv
Data variables:
    tassumpoly  (idv, proj_base, lat, lon, gwl, param) float32 650MB dask.array<chunksize=(1, 1, 40, 40, 1, 4), meta=np.ndarray>